# DKANA Thesis-Faithful Colab (v2)

Lane thesis-faithful (`thesis_faithful_dkana_v1`, accion 18D, sin Track B).

**Si falla con exit code 2:** casi siempre el clone de GitHub esta desactualizado y no trae `scripts/david_dkana_thesis_faithful_smoke.py`. Usa `REPO_SOURCE = "drive"` o sube un zip del repo actual.


## 0) Configuracion rapida

Edita solo esta celda antes de correr el resto.


In [ ]:
# --- Config (editar) ---
REPO_SOURCE = "git"  # "git" | "drive" | "upload_zip"
GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"  # cambia si tus scripts estan en otra rama

# Si REPO_SOURCE == "drive", apunta a tu copia local del repo en Drive:
DRIVE_REPO_PATH = "/content/drive/MyDrive/proyecto_grarrido_scres+ia"

# Si REPO_SOURCE == "upload_zip", sube scresia.zip a /content/ y pon:
UPLOAD_ZIP_PATH = "/content/scresia.zip"

# Modo de observacion thesis-faithful
OBSERVATION_MODE = "env_sdm_history_reward"  # o "decision_reward"
OBSERVATION_VERSION = "v5"  # solo aplica si OBSERVATION_MODE != decision_reward

# Export / entrenamiento
EPISODES = 10
MAX_STEPS = 260
SEED_START = 0
REWARD_MODE = "ReT_seq_v1"
RISK_LEVEL = "increased"
STOCHASTIC_PT = True

WINDOW_SIZE = 12
RELATION_MODE = "temporal_delta"
INCLUDE_PREV_REWARD = True

# DKANA behavior cloning
TRAIN_EPOCHS = 5
TRAIN_BATCH_SIZE = 32
TRAIN_LR = 3e-4


### Si usas `REPO_SOURCE = "drive"`

Corre antes del setup:

```python
from google.colab import drive
drive.mount('/content/drive')
```


### Si usas `REPO_SOURCE = "drive"`

Monta Drive antes del setup:

```python
from google.colab import drive
drive.mount("/content/drive")
```

## 1) Setup del repo + dependencias

Verifica que existan los scripts thesis-faithful antes de seguir.


In [ ]:
from __future__ import annotations

import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

REQUIRED_PATHS = [
    "scripts/david_dkana_thesis_faithful_smoke.py",
    "scripts/export_trajectories_for_david.py",
    "scripts/build_dkana_dataset.py",
    "scripts/train_dkana_behavior_clone.py",
    "supply_chain/dkana_env.py",
    "supply_chain/external_env_interface.py",
]


def run_cmd(cmd: list[str], *, cwd: Path | None = None) -> None:
    print("$", " ".join(cmd))
    proc = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        if proc.stderr:
            print("STDERR:
", proc.stderr)
        raise RuntimeError(f"Comando fallo (exit {proc.returncode})")


def verify_repo(root: Path) -> None:
    missing = [rel for rel in REQUIRED_PATHS if not (root / rel).exists()]
    if missing:
        raise FileNotFoundError(
            "Repo incompleto o desactualizado. Faltan:\n- "
            + "\n- ".join(missing)
            + "\n\nSolucion: usa REPO_SOURCE='drive' con tu repo actual, "
            "o sube un zip actualizado (upload_zip)."
        )


def setup_repo() -> Path:
    if REPO_SOURCE == "git":
        root = Path("/content/scresia")
        shutil.rmtree(root, ignore_errors=True)
        run_cmd(["git", "clone", "--branch", GIT_BRANCH, GIT_URL, str(root)])
        return root
    if REPO_SOURCE == "drive":
        root = Path(DRIVE_REPO_PATH)
        if not root.exists():
            raise FileNotFoundError(f"No existe DRIVE_REPO_PATH: {root}")
        return root
    if REPO_SOURCE == "upload_zip":
        root = Path("/content/scresia")
        shutil.rmtree(root, ignore_errors=True)
        root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(UPLOAD_ZIP_PATH) as zf:
            zf.extractall("/content")
        # zip puede extraer scresia/ o proyecto_grarrido.../
        candidates = [p for p in Path("/content").iterdir() if (p / "supply_chain").exists()]
        if not candidates:
            raise FileNotFoundError("Zip no contiene carpeta con supply_chain/")
        return candidates[0]
    raise ValueError(f"REPO_SOURCE invalido: {REPO_SOURCE}")


ROOT = setup_repo()
verify_repo(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
print("ROOT OK:", ROOT)
print("Python:", sys.executable)


## 2) Smoke test (contrato 18D/19D o rico)

Si el script CLI falla, esta celda tambien puede correr el smoke inline.


In [ ]:
from pathlib import Path
import subprocess

smoke_script = ROOT / "scripts/david_dkana_thesis_faithful_smoke.py"
cmd = [
    sys.executable,
    str(smoke_script),
    "--observation-mode",
    OBSERVATION_MODE,
    "--reward-mode",
    REWARD_MODE,
    "--risk-level",
    RISK_LEVEL,
    "--observation-version",
    OBSERVATION_VERSION,
    "--max-steps",
    "2",
]
if STOCHASTIC_PT:
    cmd.append("--stochastic-pt")

try:
    run_cmd(cmd, cwd=ROOT)
except Exception as exc:
    print("CLI smoke fallo, intentando inline:", exc)
    import numpy as np
    from supply_chain.external_env_interface import make_dkana_thesis_faithful_env

    env = make_dkana_thesis_faithful_env(
        reward_mode=REWARD_MODE,
        risk_level=RISK_LEVEL,
        observation_version=OBSERVATION_VERSION,
        observation_mode=OBSERVATION_MODE,
        max_steps=2,
        stochastic_pt=STOCHASTIC_PT,
    )
    obs, info = env.reset(seed=42)
    if OBSERVATION_MODE == "decision_reward":
        action = np.zeros(18, dtype=np.float32)
        action[2] = action[7] = action[12] = 1.0
        action[16] = 1.0
    else:
        raise RuntimeError("Para smoke inline rapido usa decision_reward; para otros modos arregla el repo.")
    next_obs, reward, terminated, truncated, step_info = env.step(action)
    print("inline smoke OK")
    print("action_shape", env.action_space.shape)
    print("obs_shape", obs.shape, "->", next_obs.shape)
    print("contract", step_info.get("action_contract"))
    env.close()


## 3) Export de trayectorias

`EPISODES` y `MAX_STEPS` vienen de la celda de config.


In [ ]:
mode_tag = "19d" if OBSERVATION_MODE == "decision_reward" else "sdm"
EXPORT_DIR = ROOT / "outputs" / f"data_export_thesis_dkana_{mode_tag}_colab"

export_cmd = [
    sys.executable,
    str(ROOT / "scripts/export_trajectories_for_david.py"),
    "--episodes",
    str(EPISODES),
    "--seed-start",
    str(SEED_START),
    "--reward-mode",
    REWARD_MODE,
    "--risk-level",
    RISK_LEVEL,
    "--action-contract",
    "thesis_faithful_dkana_v1",
    "--thesis-observation-mode",
    OBSERVATION_MODE,
    "--thesis-inventory-period-mode",
    "thesis_strict",
    "--max-steps",
    str(MAX_STEPS),
    "--output-dir",
    str(EXPORT_DIR),
]
if OBSERVATION_MODE != "decision_reward":
    export_cmd.extend(["--observation-version", OBSERVATION_VERSION])
if STOCHASTIC_PT:
    export_cmd.append("--stochastic-pt")

run_cmd(export_cmd, cwd=ROOT)
print("Export listo:", EXPORT_DIR)


## 4) Construir ventanas DKANA


In [ ]:
DATASET_DIR = EXPORT_DIR / f"dkana_seq_w{WINDOW_SIZE}"

build_cmd = [
    sys.executable,
    str(ROOT / "scripts/build_dkana_dataset.py"),
    "--input-dir",
    str(EXPORT_DIR),
    "--window-size",
    str(WINDOW_SIZE),
    "--relation-mode",
    RELATION_MODE,
]
if INCLUDE_PREV_REWARD:
    build_cmd.append("--include-prev-reward")

run_cmd(build_cmd, cwd=ROOT)
print("Dataset listo:", DATASET_DIR)


## 5) Verificar shapes


In [ ]:
import json
import numpy as np

row = np.load(DATASET_DIR / "dkana_row_matrices.npy")
cfg = np.load(DATASET_DIR / "dkana_config_context.npy")
act = np.load(DATASET_DIR / "dkana_action_targets.npy")
msk = np.load(DATASET_DIR / "dkana_time_mask.npy")

print("dkana_row_matrices", row.shape)
print("dkana_config_context", cfg.shape)
print("dkana_action_targets", act.shape)
print("dkana_time_mask", msk.shape)

meta = json.loads((DATASET_DIR / "metadata.json").read_text(encoding="utf-8"))
print("observation_mode:", OBSERVATION_MODE)
print("relation_mode:", meta.get("relation_mode"))
print("include_prev_reward:", meta.get("include_prev_reward"))


## 6) Entrenar DKANA (PyTorch behavior cloning)

Usa `scripts/train_dkana_behavior_clone.py` del repo.


In [ ]:
CHECKPOINT_DIR = ROOT / "outputs" / "dkana_checkpoints_colab" / f"{mode_tag}_w{WINDOW_SIZE}"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable,
    str(ROOT / "scripts/train_dkana_behavior_clone.py"),
    "--dataset-dir",
    str(DATASET_DIR),
    "--output-dir",
    str(CHECKPOINT_DIR),
    "--epochs",
    str(TRAIN_EPOCHS),
    "--batch-size",
    str(TRAIN_BATCH_SIZE),
    "--learning-rate",
    str(TRAIN_LR),
]
run_cmd(train_cmd, cwd=ROOT)

import json
metrics_path = CHECKPOINT_DIR / "training_metrics.json"
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text(encoding="utf-8")), indent=2))
print("Checkpoint:", CHECKPOINT_DIR / "dkana_policy.pt")


## 7) (Opcional) Entrenamiento inline en notebook

Si prefieres no usar subprocess para train, descomenta y corre esta celda.


In [ ]:
# Descomenta para entrenar inline:
# import json
# import numpy as np
# import torch
# from torch.utils.data import DataLoader, TensorDataset, random_split
# from supply_chain.dkana import DKANAPolicy
#
# row = np.load(DATASET_DIR / "dkana_row_matrices.npy", dtype=np.float32)
# cfg = np.load(DATASET_DIR / "dkana_config_context.npy", dtype=np.float32)
# act = np.load(DATASET_DIR / "dkana_action_targets.npy", dtype=np.float32)
# mask = np.load(DATASET_DIR / "dkana_time_mask.npy").astype(bool)
#
# ds = TensorDataset(
#     torch.from_numpy(row),
#     torch.from_numpy(cfg),
#     torch.from_numpy(mask),
#     torch.from_numpy(act),
# )
# train_ds, val_ds = random_split(ds, [int(0.8 * len(ds)), len(ds) - int(0.8 * len(ds))])
# train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=TRAIN_BATCH_SIZE)
#
# model = DKANAPolicy(
#     config_dim=cfg.shape[-1],
#     action_dim=act.shape[-1],
#     max_rows=row.shape[2] + 1,
#     max_sequence_length=row.shape[1],
# )
# opt = torch.optim.AdamW(model.parameters(), lr=TRAIN_LR)
#
# def epoch(loader, train=True):
#     model.train(train)
#     losses = []
#     for rows_b, cfg_b, mask_b, target_b in loader:
#         dist = model(rows_b, cfg_b, mask_b)
#         loss = -dist.log_prob(target_b).mean()
#         if train:
#             opt.zero_grad(set_to_none=True)
#             loss.backward()
#             opt.step()
#         losses.append(float(loss.detach()))
#     return float(np.mean(losses))
#
# for ep in range(1, TRAIN_EPOCHS + 1):
#     tr = epoch(train_loader, True)
#     va = epoch(val_loader, False)
#     print(f"epoch={ep} train={tr:.4f} val={va:.4f}")
#
# ckpt = CHECKPOINT_DIR / "dkana_policy_inline.pt"
# torch.save({"model_state_dict": model.state_dict()}, ckpt)
# print("saved", ckpt)
print("(opcional) celda inline comentada")
